# REG All — Cross-Discipline Summary of JRG DiD Estimates

This notebook compiles the main DiD results from all 11 discipline-level regression notebooks.
Each notebook estimates the same Two-Way Fixed Effects (TWFE) model with Australia, the United
Kingdom, and New Zealand as the three-country panel:

$$\log(E_{ct}) = \beta_0 + \beta_1 \cdot \text{Treated}_c + \beta_2 \cdot \text{NZ}_c + \beta_3 \cdot \text{DID}_{ct} + \sum_t \gamma_t \cdot \mathbf{1}_{[\text{year}=t]} + \varepsilon_{ct}$$

where $\text{Treated}_c = 1$ for Australia (UK and NZ are both controls), $\text{NZ}_c = 1$ for New
Zealand (UK is the level reference), $\text{Post}_t = 1$ from 2021, and
$\text{DID}_{ct} = \text{Treated}_c \times \text{Post}_t$. The coefficient $\hat{\beta}_3$
is the JRG DiD estimate: the change in AUS log-enrolments relative to the **pooled UK + NZ trend**
post-2021, expressed approximately as a percentage effect via $(\exp(\hat{\beta}_3) - 1) \times 100$.

Standard errors are HC3 heteroscedasticity-robust throughout. Two panel sizes appear:
- **N = 27 (df = 15):** 3 countries × 9 years (2016–2024)
- **N = 18 (df = 9):** 3 countries × 6 years (2019–2024), used where the JACS→CAH taxonomy break makes pre-2019 UK data irreconcilable

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.formula.api as smf
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:.4f}'.format)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent

aus_raw = pd.read_csv(ROOT / 'data' / 'clean' / 'EnrollmentsAUS_category_with_numeric_key.csv')
uk_raw  = pd.read_csv(ROOT / 'data' / 'clean' / 'uk_grouped' / 'with_categorykey' / 'UK_enrollments_grouped_comparison_all_years_with_categorykey.csv')
nz_raw  = pd.read_csv(ROOT / 'data' / 'clean' / 'NZ_bachelors_enrollments_2016_2025.csv')

year_cols = [c for c in aus_raw.columns if str(c).isdigit()]
aus_long  = aus_raw.melt(
    id_vars=['Category', 'CategoryKey'],
    value_vars=year_cols,
    var_name='year',
    value_name='enrollments',
)
aus_long['year']        = aus_long['year'].astype(int)
aus_long['enrollments'] = pd.to_numeric(aus_long['enrollments'], errors='coerce')

print('Project root:', ROOT)
print('Data files loaded.')

## 1. Compiled DiD Results

All estimates sorted by approximate percentage effect (largest negative to largest positive).


In [ ]:
DISCIPLINES = [
    # (label,                        CategoryKey, start_year, jrg_type)
    ('Management & Commerce',              8,       2016,     'Discouraged'),
    ('Others',                            11,       2019,     'Non-priority'),
    ('Engineering & Related Tech',         3,       2016,     'Priority'),
    ('Health',                             6,       2016,     'Priority'),
    ('Environment & Related',              5,       2019,     'Priority'),
    ('Information Technology',             2,       2016,     'Priority'),
    ('Architecture & Building',            4,       2016,     'Non-priority'),
    ('Society & Culture',                  9,       2019,     'Discouraged'),
    ('Creative Arts',                     10,       2016,     'Non-priority'),
    ('Natural & Physical Science',         1,       2019,     'Priority'),
    ('Education',                          7,       2016,     'Priority'),
]


def _build_panel(catkey, start):
    aus_f = aus_long[
        (aus_long['CategoryKey'] == catkey) &
        (aus_long['year'] >= start) &
        (aus_long['year'] <= 2024)
    ][['year', 'enrollments']].copy()
    aus_f['country'] = 'AUS'

    uk_f = uk_raw[uk_raw['categorykey'] == catkey].copy()
    uk_f['year'] = uk_f['AcademicYear'].str[:4].astype(int)
    uk_f = uk_f[
        (uk_f['year'] >= start) & (uk_f['year'] <= 2024)
    ][['year', 'Total UK']].rename(columns={'Total UK': 'enrollments'})
    uk_f['enrollments'] = pd.to_numeric(uk_f['enrollments'], errors='coerce')
    uk_f['country'] = 'UK'

    nz_f = nz_raw[
        (nz_raw['category_key'] == catkey) &
        (nz_raw['year'] >= start) &
        (nz_raw['year'] <= 2024)
    ][['year', 'total_bachelors']].rename(columns={'total_bachelors': 'enrollments'}).copy()
    nz_f['country'] = 'NZ'

    p = pd.concat([aus_f, uk_f, nz_f], ignore_index=True).sort_values(['country', 'year']).reset_index(drop=True)
    p['log_enrollments'] = np.log(p['enrollments'])
    p['treated']  = (p['country'] == 'AUS').astype(int)
    p['nz_dummy'] = (p['country'] == 'NZ').astype(int)
    p['post'] = (p['year'] >= 2021).astype(int)
    p['did']  = p['treated'] * p['post']
    return p


def _sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    if p < 0.10:  return '†'
    return ''


rows_raw = []
for label, catkey, start, jrg_type in DISCIPLINES:
    panel = _build_panel(catkey, start)
    m = smf.ols('log_enrollments ~ treated + nz_dummy + did + C(year)', data=panel).fit(cov_type='HC3')
    b  = m.params['did']
    se = m.bse['did']
    pv = m.pvalues['did']
    pct = (np.exp(b) - 1) * 100
    rows_raw.append({
        'Discipline': label, 'jrg_type': jrg_type,
        'beta': b, 'se': se, 'pval': pv, 'pct': pct,
        'N': int(m.nobs), 'df': int(m.df_resid), 'panel': f'{start}–2024',
    })

rows_raw.sort(key=lambda r: r['pct'])

display_rows = [{
    'Discipline': r['Discipline'],
    'β (DiD)':    round(r['beta'], 4),
    'SE (HC3)':   round(r['se'],   4),
    'p-value':    '<0.001' if r['pval'] < 0.001 else f"{r['pval']:.3f}",
    'Sig.':       _sig_stars(r['pval']),
    'Approx. %':  f"{r['pct']:+.1f}%",
    'N':          r['N'],
    'df':         r['df'],
    'Panel':      r['panel'],
    'JRG type':   r['jrg_type'],
} for r in rows_raw]

print('=== DiD Results: All Disciplines (AUS vs UK + NZ, TWFE HC3) ===')
print('Significance: *** p<0.001  ** p<0.01  * p<0.05  † p<0.10')
print()
display(pd.DataFrame(display_rows).reset_index(drop=True))

## 2. Effect Size Visualisation

Bars show the approximate percentage effect (exp(β) − 1). Disciplines significant at the
10% level are marked with a star. Colour indicates the JRG policy classification.


In [ ]:
colour_map = {
    'Priority':     '#2196F3',
    'Discouraged':  '#F44336',
    'Non-priority': '#9E9E9E',
}

labels_   = [r['Discipline'] for r in rows_raw]
pcts_     = [r['pct']        for r in rows_raw]
pvals_    = [r['pval']       for r in rows_raw]
colours_  = [colour_map[r['jrg_type']] for r in rows_raw]

fig, ax = plt.subplots(figsize=(13, 7))
bars = ax.barh(labels_, pcts_, color=colours_, alpha=0.82, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8)

for bar, p in zip(bars, pvals_):
    if p < 0.10:
        x = bar.get_width()
        offset = 0.8 if x >= 0 else -0.8
        ax.text(x + offset, bar.get_y() + bar.get_height() / 2,
                _sig_stars(p),
                va='center', ha='left' if x >= 0 else 'right', fontsize=13, color='black')

ax.set_xlabel('Approximate % change in AUS enrolments relative to UK + NZ trend (post-2021)', fontsize=11)
ax.set_title('JRG DiD Estimates by Discipline\n(AUS vs UK + NZ, TWFE OLS with HC3 robust SEs)', fontsize=13)

legend_handles = [
    mpatches.Patch(color='#2196F3', alpha=0.82, label='Priority field (JRG fee reduction)'),
    mpatches.Patch(color='#F44336', alpha=0.82, label='Discouraged field (JRG fee increase)'),
    mpatches.Patch(color='#9E9E9E', alpha=0.82, label='Non-priority / mixed'),
    mpatches.Patch(color='white', label='† p<0.10   * p<0.05   ** p<0.01   *** p<0.001'),
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Most Impacted Disciplines

### Largest negative impacts (AUS enrolments fell relative to UK)

**Management & Commerce** is the most severely impacted discipline with a **−39.4%** effect
(β = −0.5007, p < 0.001) — the only result that is statistically significant at the 1% level.
This is the single clearest empirical signal from the JRG: the large student fee increase
(+37%) and deep Commonwealth funding cut (−47.3%) directly translated into a relative
enrolment collapse in AUS compared to the UK trend. The event study shows the differential
reaching −27.5% by 2022.

**Others** (combined/general degrees) shows a large point estimate of **−29.1%** but with
very wide confidence intervals (SE = 0.331, p = 0.299). The data panel is restricted to
2019–2024 (N = 12, df = 4), and the category is difficult to interpret due to compositional
differences between AUS and UK definitions of "Others". This result cannot be treated as
reliable evidence of a JRG effect.

All other disciplines cluster between **−2.2% and −7.8%**, a band which may reflect a
modest generalised suppression effect on AUS enrolments, or simply noise given the limited
statistical power available in most models.

### Largest positive impact (AUS enrolments grew relative to UK)

**Education** shows the only clearly positive and economically meaningful effect: **+25.7%**
(β = +0.2287, p = 0.059). This is the single result where the JRG's priority-field
incentives appear to have worked as intended — Education received a large fee reduction
(≈−40% student contribution), and AUS enrolments grew substantially relative to the UK trend.
The event study shows the gap widening monotonically to +40% by 2024. At p = 0.059 the result
is marginally significant; dropping 2020 (COVID anomaly) tightens significance to p = 0.040.

**Natural & Physical Science** shows a near-zero effect (+1.3%, p = 0.599) — effectively
a null result. Both AUS and UK N&PS declined after 2021, with the differential negligible.


## 4. Statistical Reliability Assessment

The table below classifies each discipline by whether its result can be considered a
reliable representation of the JRG effect, and — where it cannot — whether the primary
reason is **insufficient data** or **no meaningful detectable effect**.

| Discipline | p-value | Verdict | Primary reason if unreliable |
|------------|---------|---------|-------------------------------|
| **Management & Commerce** | <0.001 | **Reliable — strong JRG signal** | — |
| **Education** | 0.059 | **Reliable — marginal (†)** | Borderline significance; pre-trend caveat |
| **Health** | 0.095 | **Marginally reliable (†)** | Counter-intuitive direction; pre-trend concern |
| Engineering & Related Tech | 0.227 | Not reliable | Parallel trends concern (large positive pre-trend) |
| Architecture & Building | 0.279 | Not reliable | No detectable effect; non-flat pre-trends |
| Others | 0.299 | Not reliable | Insufficient data (N=12, df=4) + category composition |
| Society & Culture | 0.389 | Not reliable | Insufficient data (N=12, df=4) |
| Creative Arts | 0.471 | Not reliable | No detectable effect; pre-existing convergence |
| Information Technology | 0.483 | Not reliable | No detectable effect; rising pre-trend |
| Environment & Related | 0.556 | Not reliable | Insufficient data (N=12, df=4) + pre-trend concern |
| Natural & Physical Science | 0.599 | Not reliable | Insufficient data (N=12, df=4); likely true null |

### Notes on each unreliable category

**Insufficient data (N = 12, df = 4):**
Four disciplines — Others, Society & Culture, Environment & Related, and Natural & Physical
Science — are restricted to a 2019–2024 panel due to irreconcilable breaks in the UK HESA
taxonomy (JACS → CAH) in 2019/20. With only 12 observations and 4 residual degrees of
freedom, confidence intervals are extremely wide and p-values have very limited power to
detect real effects. These results are inconclusive by construction, not necessarily because
the policy had no effect.

- **Others:** The category definition differs fundamentally between AUS and UK, limiting
  interpretability regardless of sample size.
- **Society & Culture:** Despite receiving the *largest* student fee increase of any
  discipline (+65%), no effect is detectable — possibly reflecting inelastic demand for
  law, psychology, and social science degrees at the margin.
- **Environment & Related:** The policy sent the strongest financial incentive (−42%
  student fee), yet the differential moved *negatively*. The pre-existing 2019 gap
  (−10.5%) suggests structural divergence pre-dating JRG — this result is confounded,
  not just underpowered.
- **Natural & Physical Science:** Likely a genuine null result — both countries declined
  similarly post-2021, consistent with sector-wide headwinds unrelated to JRG.

**No detectable effect despite adequate data (N = 18, df = 7):**
Four disciplines have full 2016–2024 panels but remain statistically insignificant.

- **Engineering & Related Tech (p = 0.227):** The large positive pre-trend (AUS already
  outperforming UK by +12% in 2019) violates the parallel trends assumption. The negative
  post-2021 estimate likely reflects mean-reversion, not a JRG effect. Inconclusive.
- **Architecture & Building (p = 0.279):** Non-flat pre-trends in 2016–2017 suggest the
  2-country parallel trends assumption is strained. The point estimate (−6.1%) is
  directionally plausible but statistically weak. Insufficient power + possible confounds.
- **Creative Arts (p = 0.471):** A pre-existing convergence in the AUS–UK differential
  from 2016–2019 accounts for much of the apparent post-JRG null. The policy likely had
  no meaningful effect on Creative Arts enrolments.
- **Information Technology (p = 0.483):** Rising pre-trend (AUS growing faster than UK
  pre-JRG). The point estimate (−7.3%) is directionally consistent with a discouraged
  or neutral field, but is statistically indistinguishable from noise.

**Health (p = 0.095 — marginal):**
Health is the most ambiguous result. The effect (−6.7%) is counter-intuitive for a
priority field where fees were cut. The estimate is stable across COVID sensitivity checks,
but the rising pre-trend (+8.3% by 2019) complicates causal attribution. The result likely
reflects a combination of a real (modest) JRG effect and residual confounding from the
asymmetric COVID shock — UK Health enrolments surged in 2020, inflating the UK baseline
and mechanically pushing the post-2021 DiD estimate negative. Treat as suggestive, not
conclusive.


## 5. Overall Summary

Of 11 disciplines analysed, **only one** (Management & Commerce) produces a statistically
significant result at the 5% level, and **two further** (Education, Health) are marginal
at the 10% level. The remaining eight are statistically inconclusive.

**What the JRG appears to have done:**
- Sharply reduced AUS enrolments in **Management & Commerce** relative to the UK+NZ trend (≈ −21.6%, p = 0.055),
  the discipline with the clearest policy signal (large fee increase + Commonwealth cut).
- Boosted **Education** enrolments (≈ +25.7%, p ≈ 0.059) relative to the UK+NZ trend, consistent with the large
  fee reduction for that discipline.

**What remains uncertain:**
- Whether the modest estimates across Engineering, IT, Architecture, Creative Arts,
  and Health represent real (small) policy effects or noise.
- The four restricted-panel disciplines (N = 18, df = 9) are simply underpowered; their results
  neither confirm nor rule out a JRG effect.

---

## Policy Alignment Check

| Discipline | Policy intent | Observed direction | Aligned? |
|---|---|---|---|
| Management & Commerce | Discourage (fees ↑ +37%) | Negative (≈ −21.6%) | ✓ Strong |
| Education | Encourage (fees ↓ ≈ −40%) | Positive (≈ +25.7%) | ✓ Strong |
| Information Technology | Encourage (fees ↓) | Positive (≈ +15.1%) | ✓ Direction only |
| Engineering & Related Tech | Encourage (fees ↓) | Positive (≈ +1.9%) | ✓ Direction only |
| Others | Non-priority (fees ↑) | Negative (≈ −23.5%) | ✓ Direction only (inconclusive) |
| Society & Culture | Discourage (fees ↑ +65%) | Positive (≈ +0.8%) | ✗ Unexpected (effectively zero) |
| Health | Encourage (fees ↓) | Negative (≈ −1.1%) | ✗ Unexpected (effectively zero) |
| Environment & Related | Encourage (fees ↓ ≈ −44%) | Negative (≈ −6.5%) | ✗ Unexpected |
| Natural & Physical Science | Encourage (fees ↓) | Near-zero (≈ +0.9%) | ~ Null |
| Architecture & Building | Non-priority | Negative (≈ −5.2%) | — |
| Creative Arts | Non-priority | Negative (≈ −2.2%) | — |

> *All effects are from the 3-country AUS vs UK + NZ TWFE specification (HC3 SEs). None of the directional results above are statistically significant unless otherwise noted.*

### Summary

**Affected in the expected direction** — two disciplines show both statistical signal and policy alignment. **Management & Commerce** (discouraged field) is the clearest result in the study: a fee increase plus Commonwealth cut produced a large, near-significant relative enrolment decline (≈ −21.6%, p = 0.055). **Education** (priority field) is the only priority discipline where the intended positive response is detectable (≈ +25.7%, p ≈ 0.059). Two further priority fields — **IT** (+15.1%) and **Engineering** (+1.9%) — point in the right direction but fall well short of significance.

**Affected in the unexpected direction** — **Environment & Related** is the most anomalous result: it received the steepest student fee cut of any field (≈ −44%) yet AUS enrolments declined relative to the UK+NZ trend (≈ −6.5%). A pre-existing negative differential in 2019 suggests structural divergence that predates the policy, making causal attribution difficult. **Health** and **Society & Culture** are technically in the wrong direction, but both estimates are so close to zero (≈ −1.1% and +0.8% respectively) that they are more accurately characterised as null results than genuine reversals.

**Cannot draw conclusions** — five disciplines are inconclusive for different reasons. **N&PS**, **S&C**, **Environment**, and **Others** are restricted to 2019–2024 panels (N = 18, df = 9), leaving insufficient power to detect modest effects. **Architecture & Building** and **Creative Arts** have adequate panel length but show small, statistically weak estimates alongside pre-trend concerns. Across this group, the data neither confirms nor rules out a JRG effect — the absence of statistical significance should not be read as evidence the policy did nothing.